# 10. Lexical Term Accuracy & Rare-Word Precision Study
Analyzes the master lexical corpus itself, then demonstrates the rare-word / terminology evaluation machinery on synthetic example translations (real model predictions come from notebook 09's saved output on the actual training/inference environment).

In [ ]:
# ============================================================
# PATH BOOSTER — Guarantees project root in sys.path & CWD
# ============================================================
import os, sys
try:
    cwd = os.getcwd()
except FileNotFoundError:
    cwd = os.path.expanduser('~')
    os.chdir(cwd)
proj_dir = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(proj_dir):
    os.chdir(proj_dir)
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


In [1]:
import os

if "COLAB_GPU" in os.environ or os.environ.get("COLAB_RELEASE_TAG"):
    if not os.path.exists("Ekegusii-LLM-Translation"):
        os.system("git clone https://github.com/aykahsay/Ekegusii-LLM-Translation.git")
    os.chdir("Ekegusii-LLM-Translation")
    os.system("pip install -q -r requirements.txt")
elif not os.path.exists("src") and os.path.basename(os.getcwd()) == "notebooks":
    # Running locally via `jupyter nbconvert` from within notebooks/ --
    # the repo root (containing src/, data/) is one directory up.
    os.chdir("..")

import sys
sys.path.insert(0, os.getcwd())

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")


In [2]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.master_corpus.manager import MasterCorpusManager

manager = MasterCorpusManager()
lexical_df = manager.load_lexical_corpus()
print(f'{len(lexical_df)} lexical entries')
lexical_df.head()

INFO | Loaded Master Lexical Corpus: 268 terms.


268 lexical entries


,lexicon_id,English,Kiswahili,Ekegusii,source,dataset_origin
0,1,NaN,jambo,amangʼana,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
1,2,NaN,rafiki,omosani,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
2,3,NaN,mtu,omonto,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
3,4,NaN,watu,"abanto, abantu",Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv
4,5,NaN,mume,omosacha,Dictionary,Online_Glosbe_Swahili_Ekegusii_Dictionary.csv


## Column coverage
See `docs/datasets.md`: English is currently 0% populated.

In [3]:
for lang in ['English', 'Kiswahili', 'Ekegusii']:
    non_null = lexical_df[lang].notna().sum()
    print(f'{lang}: {non_null}/{len(lexical_df)} ({100*non_null/len(lexical_df):.1f}%)')

English: 0/268 (0.0%)
Kiswahili: 268/268 (100.0%)
Ekegusii: 268/268 (100.0%)


## Rare-word identification on the sentence corpus

In [4]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.tokenizer.rare_words import RareWordIdentifier

sentence_df = manager.load_sentence_corpus().sample(2000, random_state=42)
eke_sentences = sentence_df['Ekegusii'].dropna().astype(str).tolist()

identifier = RareWordIdentifier()
rare_words = identifier.identify_rare_words(eke_sentences, max_frequency=2)
print(f'{len(rare_words)} rare Ekegusii word types (freq <= 2) in this sample')
rare_words[:20]

C:\Users\Admin\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO | Loaded Master Sentence Corpus: 49,277 concepts.


INFO | Identified 7,967 rare word type(s) (freq <= 2).


7967 rare Ekegusii word types (freq <= 2) in this sample


['000gosimekwa',
 '100',
 '10th',
 '110',
 '112',
 '116',
 '1200',
 '121',
 '127',
 '128',
 '12na',
 '133',
 '13na',
 '1400',
 '146',
 '147',
 '14na',
 '150',
 '15na',
 '16na']

## Terminology consistency checker (demo)

In [5]:
import os, sys
p = os.path.join(os.path.expanduser('~'), 'Ekegusii-LLM-Translation-main')
if os.path.isdir(p) and p not in sys.path: sys.path.insert(0, p); os.chdir(p)

from src.evaluation.terminology import TerminologyConsistencyChecker

checker = TerminologyConsistencyChecker.build_from_lexical_corpus(
    lexical_df, source_lang='Kiswahili', target_lang='Ekegusii', min_term_length=4
)
print(f'{len(checker.terminology_map)} curated terms loaded from the lexical corpus')

124 curated terms loaded from the lexical corpus
